# VQCC Hedge Wizardry

**What this is.** An ongoing public collection of *small* verified tweaks to circuits for the [verifiable quantum circuit compilation competition](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation). A tweak lands here when it passes the local unmodified verifier and its expected gain is below +0.5% of the overall score. These gains are released straight away: the circuit in the [VQCC Hedge Wizardry dataset](https://www.kaggle.com/datasets/manwithacat/vqcc-hedge-wizardry), the explanation here. The notebook then runs the verifier on Kaggle.

**How releases work.** Each new version of this notebook opens with what is new in it: the entry, the gain, and the technique it demonstrates, explained in full. Earlier entries are then summarised in a few lines each, with a link to the notebook version where they were the headline, for the full detail. A techniques index lists every technique shown so far. The [version history](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) of this notebook and the dataset is the dated record.

Each entry is:

- **explained**: what changed and why it pays, written for someone new to the task;
- **verified**: the unchanged official verifier passes the circuit, on Kaggle, in the cells at the end. For DQI the gain is also judged on its expected cost over 48 fixed random draws, never on one lucky draw;
- **credited**: most tweaks are made on someone else's public circuit, and that author is named first, with the licence kept.

**It is also a working submission notebook.** The hidden cells at the end take the cheapest public circuit for each task from our [VQCC public circuits dataset](https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits). Where an entry is cheaper for its task, they swap it in, check every file's SHA-256, run the unchanged official verifiers and write `submission.zip` with a receipt.

**How to read it.** The code is hidden; click a cell to see it. Numbers carry the evidence tier they were measured at, and a number is never shown at a higher tier than that:

- *static model*: arithmetic on the official cost formula;
- *local prescreen*: our re-implementation of the official evaluator's functions, used inside a search;
- *local engine*: our fast re-implementation of the simulator, cross-checked against the official verifier;
- *local official verifier*: the unmodified competition verifier on a local build of kickmix (*re-verified by us* when the circuit is someone else's published one);
- *official verifier (Kaggle)*: the same verifier run by this notebook on Kaggle.

Bigger changes, and the release-and-recompute method, are explained in [Quantum Voodoo](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo).

In [ ]:
def rank_cost(r):
    # Cost used to rank a manifest row within its task (lower is better); None if unscored. A DQI cost at one salt is
    # one draw of its sampled failures (one or two phase errors move R from 1.19126 to 1.19673 / 1.20139), so DQI
    # ranks by dqi_expected_cost (mean over a 48-draw sweep) when known, else dqi_zero_failure_cost (R floor x Q^2 x
    # mean Toffolis), else local_task_score (older manifests). invsqrt and chemistry rank by local_task_score.
    keys = ("dqi_expected_cost", "dqi_zero_failure_cost", "local_task_score") if r.get("task") == "dqi" \
        else ("local_task_score",)
    for k in keys:
        try:
            v = float(r.get(k) or "nan")
        except ValueError:
            continue
        if v == v:
            return v
    return None

def rank_basis(r):
    # Which column rank_cost used, for the tables
    if r.get("task") != "dqi":
        return "default salt"
    return ("expected (48 draws)" if r.get("dqi_expected_cost") else
            "zero-failure" if r.get("dqi_zero_failure_cost") else "default salt")

# Tie rule: rows within REL_TIE (relative) of the task's cheapest ranking cost are a tie, and the tie goes to the
# earliest public circuit (the original). A byte-different copy of a circuit changes the verifier's measurement
# draw, so it can come out a hair cheaper by noise alone; it should not displace the original's designer.
import datetime as _dt
REL_TIE = 1e-5

def public_when(r):
    # First non-empty, readable public_since / first_known / release_date as UTC (date only = 00:00 UTC); else None
    for k in ("public_since", "first_known", "release_date"):
        s = (r.get(k) or "").strip()
        if not s:
            continue
        try:
            d = _dt.datetime.fromisoformat(s.replace("Z", "+00:00"))
        except ValueError:
            continue
        return d.replace(tzinfo=_dt.timezone.utc) if d.tzinfo is None else d.astimezone(_dt.timezone.utc)
    return None

def rank_order(rows):
    # Scored rows of one task: the tie group (cost <= cheapest x (1 + REL_TIE)) earliest public first (missing dates
    # last, then lowest cost, then sha256), then the rest by cost, then sha256
    scored = [(rank_cost(r), r) for r in rows]
    scored = [(c, r) for c, r in scored if c is not None]
    if not scored:
        return []
    limit = min(c for c, _ in scored) * (1 + REL_TIE)
    far = _dt.datetime.max.replace(tzinfo=_dt.timezone.utc)
    def tie_key(cr):
        w = public_when(cr[1])
        return (w is None, w or far, cr[0], cr[1].get("sha256") or cr[1].get("sha12") or "")
    tied = sorted((cr for cr in scored if cr[0] <= limit), key=tie_key)
    rest = sorted((cr for cr in scored if cr[0] > limit),
                  key=lambda cr: (cr[0], cr[1].get("sha256") or cr[1].get("sha12") or ""))
    return [r for _, r in tied + rest]

def pick_best(rows_for_task):
    # The row to use for a task: rank_order()[0], or None if no row is scored
    order = rank_order(rows_for_task)
    return order[0] if order else None


import base64, csv, glob, os, hashlib, json, shutil, sys, time, zipfile
from math import prod
from pathlib import Path

T0 = time.monotonic()
import numpy as np
import pandas as pd
from IPython.display import HTML, display

TASK_NAMES = {"invsqrt": "Inverse square root", "dqi": "DQI decoder", "chemistry": "Chemistry (dfthcsos)"}

def show(df, height=None):
    # A scrollable HTML table: readable on the notebook page, with live links
    html = df.to_html(index=False, render_links=True, escape=True, float_format=lambda x: f"{x:,.6g}")
    style = f"max-height:{height}px;overflow:auto;" if height else "overflow:auto;"
    display(HTML(f'<div style="{style}">{html}</div>'))

INPUT = Path(os.environ.get("VQCC_NB_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("VQCC_NB_WORK", "/kaggle/working"))
SUB = Path(os.environ.get("VQCC_NB_SUB", "/tmp/submission"))   # assembled here; only submission.zip and check.json go to Output
WHEEL_GLOB = "kickmix-*-manylinux*x86_64*.whl"
# Organisers' reference costs (the score is the geometric mean of reference / ours over the three tasks)
REFERENCE = {"invsqrt": 2076774.233715282, "dqi": 24439463583672.93, "chemistry": 3355599588.3155375}
VERIFIER_TASK = {"invsqrt": "invsqrt", "dqi": "dqi", "chemistry": "dfthcsos"}

def find(pattern, root=INPUT):
    hits = sorted(Path(p) for p in glob.glob(f"{root}/**/{pattern}", recursive=True))  # follows symlinked mounts
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {root}")
    return hits

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()

def family_sha256(directory):
    # The dataset's chemistry family hash: SHA-256 of sorted '<relative path>\t<file sha256>\n' lines
    files = sorted(p for p in Path(directory).rglob("*") if p.is_file())
    text = "".join(f"{p.relative_to(directory).as_posix()}\t{sha256(p)}\n" for p in files)
    return hashlib.sha256(text.encode()).hexdigest()

In [ ]:
# Figures live in the Hedge Wizardry dataset's figures/ folder.
def figure(name, caption=""):
    hits = sorted(Path(p) for p in glob.glob(f"{INPUT}/**/figures/{name}", recursive=True))
    if not hits:
        print(f"[figure {name} is not in the attached dataset version]")
        return
    mime = "image/gif" if name.endswith(".gif") else "image/png"
    data = base64.b64encode(hits[0].read_bytes()).decode()
    cap = f'<figcaption style="color:#52514e;font-size:13px;margin-top:4px">{caption}</figcaption>' if caption else ""
    display(HTML(f'<figure style="margin:10px 0"><img src="data:{mime};base64,{data}" '
                 f'style="max-width:100%;height:auto" alt="{name}"/>{cap}</figure>'))

## 1. New in this release (Version 10)

**HW-09 (2026-11-02, dqi): conditionally clean S12/S6 flag computations on busyaprime’s public DQI.** Twelve 12-input AND computations and twelve 6-input AND computations in busyaprime's public decoder are replaced with our exhaustively checked conditionally clean templates. All 24 sites have proved clean-wire obligations. Same 892 qubits; -293.998 mean Toffolis per shot; -0.07351% DQI expected cost on 48 shared draws, cheaper on every draw with identical errors. Expected portfolio GM 23.90474283 to 23.91060278 (+0.02451%, arithmetic). Unmodified local official verifier PASS; hosted gate pending.

Techniques demonstrated:

- **Conditionally clean flag computation with measurement-based cleanup**: A clean target and one or two clean scratch wires replace a 6- or 12-input dirty-borrow compute chain; input wires become scratch only in the branch where their values are known, are restored, and temporary products are measured with phase corrections.
- *Reused:* Measurement-based uncomputation of a borrowed-wire AND (see the techniques index).

The full write-up follows: what improved and by how much, the technique explained, what it builds on, and how to reproduce or use the files. Earlier entries are summarised in section 2, with links to the notebook version where each was the headline.

### HW-09: conditionally clean flag computations on busyaprime's 892-qubit DQI

The decoder repeatedly asks whether a word or half-word is zero. After a change of basis, these tests become ANDs of twelve or six bits. The public circuit computes them using chains that borrow wires holding other values and restore those values afterwards. Those chains cost 36 and 14 Toffolis respectively, even though the flag target and two nearby scratch wires are clean at the twelve-input sites, and the flag and one scratch wire are clean at the six-input sites.

Our change uses those clean wires to start a construction with **conditionally clean ancillae**. A wire need only be clean in the branch where it is used; the controls make its other branch irrelevant. Partial products are then erased by measurement-based uncomputation, with phase corrections conditioned on the measurement outcomes. The twelve-input construction costs 19 static Toffoli-class gates, 18 expected; the six-input construction costs 8 static, 7.5 expected. Every borrowed wire is restored. This changes the flag computations; the public base already incorporates HW-07's flag uncomputation.

| Per epoch | Public base | HW-09 expected | Saving |
| --- | ---: | ---: | ---: |
| Twelve-input AND | 36 | 18 | 18 |
| Six-input AND | 14 | 7.5 | 6.5 |
| Twelve epochs | 600 | 306 | 294 |

**How we found and checked the sites.** `qhdl census --class cc-chain` found all 24 sites directly in the circuit. It checks each block's Boolean action over every value of its inputs and borrowed wires, and proves the target and required scratch wires are zero at entry. All 24 sites had proved obligations; no sampled clean-wire assumption was used. The two replacement templates separately check all 4,096 and 64 input assignments, including each measurement's phase correction. `qhdl rewrite` applies the sites and emits an edit map and certificate. The whole-circuit checks below cover the interaction with the decoder.

| Measurement | Public `92750bd64525` | HW-09 `1698e864a3a6` |
| --- | ---: | ---: |
| Qubits | 892 | 892 |
| Mean Toffolis per shot, 48 fixed draws | 399,969.899 | 399,675.901 |
| Mean DQI cost, same draws | 379,362,607,952.24 | 379,083,756,899.51 |
| Draws with an error | 7 | 7 |
| Expected portfolio GM, arithmetic | 23.90474283 | 23.91060278 |

The DQI cost change is -0.07351%, giving +0.02451% expected portfolio GM with the other public components fixed. These are local-engine measurements over 48 shared draws and arithmetic on them, not a leaderboard score. The unchanged official verifier also passed locally at the default salt (cost 378,829,899,786.41); the hosted entry gate below independently checks these exact bytes. The default salt uses a circuit-hash seed, so its cost must not be compared with the base's different default-salt draw.

**Credit.** Base by **Akmal Xodarev (Busya PRIME)**: [33 QCC V8 members](https://www.kaggle.com/datasets/busyaprime/33-qcc-v8-members) v9, from [Exact DQI Rewrites](https://www.kaggle.com/code/busyaprime/23-910-local-gm-exact-dqi-rewrites-892-qubits) v15, Apache-2.0. The core decoder is **SomeJoint (Agniv Banerjee)**'s q26 892-qubit design; busyaprime adds exact product caching, credits **Ahmed Berat Özer** for that idea, and incorporates our HW-07 ladder. Our S12/S6 templates were developed in the earlier local relay, following **Khattar and Gidney**'s conditionally clean ancilla method; this port and verification are by **manwithacat (James Barlow), assisted by Codex (OpenAI)**.

**Files and reuse.** Copy `circuits/dqi/1698e864a3a6.kmb` as `dqi.kmb`. SHA-256: `1698e864a3a638ef0569c438f01958731ebc164a917a4b84b8ff14598b83ab54`. The dataset includes `evidence/HW-09/draws_48.csv` for the paired comparison and `evidence/HW-09/verify_summary.json` for the local official verifier. The source and modifications retain Apache-2.0 notices. The method follows Khattar and Gidney's conditionally clean ancilla constructions; our earlier local relay developed the S12/S6 templates, and this release applies them mechanically to the newer public frontier.

## 2. Earlier releases

Each earlier entry in brief: the gain at the time, the technique, the credit, and whether a newer public circuit has since overtaken it. Its circuit stays in the dataset, and the entry gate at the end still verifies it. For the full explanation, open the notebook version where it was the headline.

### HW-08, headline of Version 8 (2026-11-01, dqi): measurement-based uncomputation for the 12 three-input flag ANDs and the 12 six-input normalize flags, ported to SomeJoint's 892-qubit DQI

SomeJoint's (Agniv Banerjee) newer public 892-qubit DQI decoder `b7ca9bce749d` overtook HW-07 on the public frontier. It still clears its 12 three-input flag ANDs and 12 six-input normalize flags by re-running borrowed-wire Toffoli chains, so we carried both HW-06's and HW-07's measurement-based uncomputation onto it. Our census tool found all 24 sites in about a minute. Result `5e86f69a9611`: same 892 qubits, −207.01 mean Toffolis per shot, −0.0514% DQI cost over 48 fixed draws, cheaper on every draw, with the same 7 failing shots (local engine); about +0.017% expected overall score. Local official verifier PASS at the default salt. **Published as a reference record:** before release, busyaprime's public exact product caching on the same SomeJoint base (`3779a3431e9c`) overtook it by 0.60% on the same 48 draws; that circuit already clears the HW-06-class sites by measurement-based uncomputation, and our census no longer finds the HW-07-class sites there. The value of this entry is the worked port: what the circuit does, which sites changed, and the steps taken.

- **Technique:** Porting a tweak to a new base by site census.
- **Credit:** base circuit `b7ca9bce749d` by **SomeJoint (Agniv Banerjee)** ([notebook](https://www.kaggle.com/code/somejoint/quantum-circuits-23-85-local-gm-892-qubit-dqi) v26, Apache-2.0): GPT-6 Astra Pro's q26 decoder round, with Claude putting SomeJoint's flag carry, which follows **Ali Haydar Özdağ**'s, on it; the flag carry is as in `bdee18489fac` (**yoyobar**'s one-carry history). Change: HW-06's and HW-07's measurement-based uncomputation at 24 sites, found by our census tool, by manwithacat.
- **Status (2026-11-02):** Headline of Version 8, published as a reference record. Overtaken before release by busyaprime's public exact product caching on the same SomeJoint base, `3779a3431e9c` (48-draw mean 379,530,489,015; -0.60% against HW-08 on the same draws), which also clears the HW-06-class sites (12 self-uncompute Toffolis) by measurement-based uncomputation; our census finds the HW-07-class sites no longer present there. Local official verifier PASS at the default salt; the entry gate in this version is its first hosted run.
- **Full detail:** [Version 8](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/8) of this notebook.

### HW-07, headline of Version 7 (2026-11-01, dqi): measurement-based uncomputation with a two-wire ladder for the 12 six-input normalize flags in yoyobar's 892-qubit DQI

On our HW-06 circuit (yoyobar's public 892-qubit DQI `6d58bd6d0232` with 12 flag ANDs made measurement-based), we clear the six-input pivot flag of each epoch's normalize step by measurement-based uncomputation instead of a 16-Toffoli chain on borrowed wires. When the measurement gives 1, the phase fix rebuilds the six-input AND on a ladder of two clean wires, about 2.25 Toffolis expected per site. Result `1779b4e5967f`: same 892 qubits, −164.95 mean Toffolis per shot, −0.0409% DQI cost over 48 fixed draws against HW-06 and −0.0513% against `6d58bd6d0232`, with the same 7 failing shots (local engine); about +0.014% expected overall score against HW-06 (+0.017% against `6d58bd6d0232`). Local official verifier PASS at the default salt.

- **Technique:** Measurement-based uncomputation of a multi-input AND with a clean-wire ladder.
- **Credit:** base circuit `c253111974fc`, our HW-06, on **yoyobar**'s `6d58bd6d0232` (with GPT-6 Astra; [notebook](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits) v11, Apache-2.0), the infinity-lead product reuse on **SomeJoint (Agniv Banerjee)**'s 892-qubit flag-carry circuit `bdee18489fac` ([notebook](https://www.kaggle.com/code/somejoint/quantum-circuits-23-77-local-gm-892-qubit-dqi) v25, Apache-2.0), whose flag carry follows **Ali Haydar Özdağ**'s flag carry. Reading Ali Haydar Özdağ's (ahoirg) public reproducible rebuild of SomeJoint's decoder sources ([notebook](https://www.kaggle.com/code/ahoirg/score-23-64254-reproducible-circuit-portfolio) v1) explained the normalize step's wires. Change: the two-wire measurement-based uncomputation of the 12 six-input flags, by manwithacat.
- **Status (2026-11-02):** Published on 2026-11-01: the notebook was pushed twice with identical content, as Versions 6 and 7 (a duplicate push), so Version 7 is its headline. Hosted official verifier PASS at the default salt in Version 6 (384,137,374,413.52, equal to `entries.csv`). Overtaken on the public frontier the same day by SomeJoint's (Agniv Banerjee) `b7ca9bce749d`: 48-draw mean cost 382,020,234,789.40 against HW-07's 382,637,339,169.98 (local engine, same draws). HW-08 carries HW-06's and HW-07's techniques onto `b7ca9bce749d`.
- **Full detail:** [Version 7](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/7) of this notebook.

### HW-06, headline of Version 5 (2026-11-01, dqi): measurement-based uncomputation for the 12 borrowed-wire flag ANDs in yoyobar's 892-qubit DQI

On yoyobar's public 892-qubit DQI circuit `6d58bd6d0232` (on SomeJoint's flag carry), our measurement-based uncomputation replaces 12 four-Toffoli uncomputes that borrowed an in-use wire. Result `c253111974fc`: same 892 qubits, −42.0 mean Toffolis per shot, −0.0104% DQI cost over 48 fixed draws with the same 7 failing shots (local engine), about +0.0035% expected overall score. Local official verifier PASS at the default salt.

- **Technique:** Measurement-based uncomputation of a borrowed-wire AND; Sound value tracking for provably-zero targets.
- **Credit:** base circuit `6d58bd6d0232` by **yoyobar** (with GPT-6 Astra; [notebook](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits) v11, Apache-2.0), the infinity-lead product reuse on **SomeJoint (Agniv Banerjee)**'s 892-qubit flag-carry circuit `bdee18489fac` ([notebook](https://www.kaggle.com/code/somejoint/quantum-circuits-23-77-local-gm-892-qubit-dqi) v25, Apache-2.0), whose flag carry follows **Ali Haydar Özdağ**'s flag carry. Change: measurement-based uncomputation at the 12 borrowed-wire flag ANDs, and the value tracker that found them, by manwithacat.
- **Status (2026-11-02):** Published on 2026-11-01 as the headline of Version 5; its hosted run of the unchanged official verifier on Kaggle passed (cost 382,537,501,561.76, 0 errors, equal to `entries.csv`), and it was that version's DQI pick. Superseded for DQI by HW-07 (`1779b4e5967f`), which builds on it: −0.0409% DQI cost over the same 48 draws (local engine).
- **Full detail:** [Version 5](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) (open the version list and choose Version 5) of this notebook.

### HW-05, headline of Version 4 (2026-11-01, chemistry): C40 open-pattern chemistry: fixed-word constrained refit

On Lera’s (marculera) public open-pattern C40 family `b163954eb5ac`, our weights-only constrained refit lowered λ_eff 47.105858 → 47.048390. The local unmodified verifier passed at 326,959,774.66 versus 327,359,144.12 (−0.1220% chemistry cost), about +0.0407% overall score by static arithmetic. The 8-bit words and all 15 angle loaders are byte-identical to Lera’s.

- **Technique:** Fixed-e1 constrained refit on an open-pattern C40 table.
- **Credit:** C40 factorization, 8-bit open-pattern words and all 15 loaders by **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v9, Apache-2.0). Change: the fixed-word constrained W/H2_I refit by manwithacat (James Barlow), assisted by Codex (OpenAI). The refit method was developed in HW-03 and applied to C42 in HW-04.
- **Status (2026-11-02):** Published and hosted-verifier PASS on 2026-11-01. Overtaken by Heliosli’s public chemistry: `0d75ed93ce40` (326,934,192.97) in the same hosted run, then `ffcc1ee812ac` (326,902,029.99, Heliosli’s notebook v36), and a public parts assembly of Heliosli’s `ffcc1ee812ac` weights with angle loaders that include Dmitrii Gluzdov’s (326,821,497.37). All three costs are our local official verifier runs; HW-05 is 326,959,774.66.
- **Full detail:** [Version 4](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook?scriptVersionId=364542669) of this notebook.

### HW-04, headline of Version 3 (2026-10-31, chemistry): C42 chemistry weights: refit spending e1 slack, alternated with the column gauge

Weights only, on Lera's (marculera) C42 table `e9a9289ab06e`: our refit spent the unused e2 and E_gap margin and raised e1 in steps to 19.9999 mHa, alternated with the column gauge. λ_eff 47.080918 → 46.976679, −0.2214% chemistry cost (local official verifier PASS), about +0.074% overall score. Angle loaders and word table byte-identical to Lera's.

- **Technique:** Slack-spending constrained refit (stepped e1 continuation); KKT shadow prices of the error margins.
- **Credit:** C42 factor/word table and loaders by **Lera (marculera)** (e9a9289ab06e, [notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v6, Apache-2.0). Change: the e1-spending refit + gauge alternation, by manwithacat. (The method was first run on Ahmed Berat Özer's C43 polish e9eb9336a5ec.)
- **Status (2026-11-02):** Overtaken by Lera’s public C40 open-pattern family `b163954eb5ac` and this HW-05 refit (local verifier cost 326,959,774.66). HW-04 remains a separate C42 demonstration.
- **Full detail:** [Version 3](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) (open the version list and choose Version 3) of this notebook.

### HW-03, headline of Version 2 (2026-10-31, chemistry): chemistry weights: constrained refit alternated with the column gauge

Weights only, on the public chemistry family `ec11a966fb64` (Ahmed Berat Özer's weights in Isabe11aM00re's per-rank column gauge): our constrained refit alternated with the column gauge took λ_eff from 48.293182 to 48.274616, −0.0384% chemistry cost (local official verifier PASS), about +0.013% overall score. Angle loaders and word table byte-identical.

- **Technique:** Constrained λ_eff refit at fixed words; Refit alternated with a per-rank orthogonal column gauge.
- **Credit:** weights by **Ahmed Berat Özer** (4eabe756bc36, [notebook](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression), Apache-2.0); per-rank column-gauge idea and the base point ec11a966fb64 by **Isabe11aM00re** ([notebook](https://www.kaggle.com/code/isabe11am00re/qcc-v47-submission) v3, Apache-2.0); word table by **Atakan Aldemir**; the (8, 143, 60) factorization by **SomeJoint (Agniv Banerjee)**; loaders by **Heliosli, Dmitrii Gluzdov and Atakan Aldemir**. Change: the refit + gauge alternation and the gauge search, by manwithacat.
- **Status (2026-11-02):** Overtaken: Lera's (marculera) C42 table `e9a9289ab06e` (331,760,020, re-verified by us) is 1.7% cheaper than HW-03, and HW-04 (section 1) applies the same method to it.
- **Full detail:** [Version 2](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) (open the version list and choose Version 2) of this notebook.

### HW-01, headline of Version 1 (2026-10-30, dqi): 889 -> 888 qubits on yoyobar's DQI by release-and-recompute

On yoyobar's public 889-qubit DQI decoder `73e1b187b288`, our release-and-recompute freed one qubit for +460.4 mean Toffolis per shot (a qubit is worth about 960 here). Result `5faba4591853`: 888 qubits, −0.117% DQI cost at the default salt and −0.108% over 48 fixed draws (local official verifier; local engine), about +0.036% expected overall score. One preregistered threshold was missed (8 failing draws of 48 against the base's 7) and the release said so.

- **Technique:** Release-and-recompute (spooky pebbling); Liveness ledger to find the idle hold.
- **Credit:** base circuit `73e1b187b288` by **yoyobar** ([notebook](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits) v8, Apache-2.0), on the decoder line of **SomeJoint (Agniv Banerjee)**; releasing a pole flag and recomputing it was first used publicly by **heliosli** on SomeJoint's decoder. Change: release-and-recompute (Quantum Voodoo line), the q852 release, the q878 reset and the handover erase, by manwithacat.
- **Status (2026-11-02):** Overtaken: newer public 888-qubit DQI circuits are cheaper, for example yoyobar's `25db915d86ab` and `e20e6abea7e2` and Ahmed Berat Özer's `fe98677c61ef` (398,268,953,061 at the default salt, −0.93% against HW-01; re-verified by us with the local official verifier, 2026-10-31).
- **Full detail:** [Version 1](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook?scriptVersionId=364036151) of this notebook.

## 3. Techniques index

Every technique demonstrated so far, where it first appeared, and which later entries reuse it.

| Technique | First shown | Reused in | What it does |
| --- | --- | --- | --- |
| Release-and-recompute (spooky pebbling) | HW-01, [Version 1](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook?scriptVersionId=364036151) | — | Measurement-based uncomputation (HMR) releases a value that sits idle across every peak run; it is recomputed (rematerialised) where it is next needed and the phase correction applied, so the peak qubit count falls for a few hundred extra Toffolis. Credit: ours (Quantum Voodoo line); releasing a pole flag was first used publicly by heliosli. |
| Liveness ledger to find the idle hold | HW-01, [Version 1](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook?scriptVersionId=364036151) | — | A liveness analysis (render.py ledger) lists each value's idle time at peak width, priced at the qubit's shadow price; it named the target (q852) before anything was built. Credit: ours (kickmix-macos fork). |
| Constrained λ_eff refit at fixed words | HW-03, [Version 2](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) | HW-04, HW-05 | Projected L-BFGS on log λ_eff over the continuous weights and offsets, with e1, e2 and E_gap held as equality constraints just inside their limits and a Gauss–Newton step back onto them after every move; the angle words and angle loaders, so qubits and Toffolis, never change. Credit: ours (constraint-manifold retraction idea from anazemcev's projected L-BFGS). |
| Refit alternated with a per-rank orthogonal column gauge | HW-03, [Version 2](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) | HW-04 | Rotating one rank's columns by an orthogonal matrix leaves e1, e2 and E_gap unchanged but changes the column L1 norms in Λ; exact Jacobi sweeps plus a smoothed-L1 L-BFGS find a sparser frame between refits. Credit: gauge idea by Isabe11aM00re; the alternation and gauge search ours. |
| Slack-spending constrained refit (stepped e1 continuation) | HW-04, [Version 3](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) | — | From a point that leaves error margin unused, the refit spends the e2 and E_gap margin and raises the e1 target in steps (16.5 → 18 → 19 → 19.9999 mHa), each from the previous point, with column gauges between; the extra e1 room lets the offsets lower the one-body norm Σ|eig h1′|. Credit: ours. |
| KKT shadow prices of the error margins | HW-04, [Version 3](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) | — | The Lagrange multipliers of the equality-held constraints price each margin in % of λ_eff per mHa, showing which margin pays (here e2, about 200 times e1 at the final point) before compute is spent. Credit: ours (standard sensitivity analysis). |
| Fixed-e1 constrained refit on an open-pattern C40 table | HW-05, [Version 4](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook?scriptVersionId=364542669) | — | Move continuous W and H2_I on the e1/e2/E_gap constraint surface while freezing Lera’s integer words and all compiled loaders; a nonzero projected KKT residual predicts local room to improve λ_eff. Credit: our application of the HW-03 constrained refit to Lera’s C40 table. |
| Measurement-based uncomputation of a borrowed-wire AND | HW-06, [Version 5](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) | HW-07, HW-08, HW-09 | A three-input AND computed with the help of a borrowed (dirty) wire costs 4 Toffolis to uncompute the same way. Measuring the result in the X basis instead clears it for free; the phase it leaves, (−1)^(m·abc), is removed by one CCZ on the three inputs, run only when the measured bit m is 1 (about half the shots). Credit: the construction is Gidney's (2018) measurement-based uncomputation of a temporary logical-AND; its application to these 12 sites is ours. |
| Sound value tracking for provably-zero targets | HW-06, [Version 5](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/versions) | — | A conservative tracker follows which wires are certainly 0 after each operation. A Toffoli whose target is provably 0 just afterwards is an uncompute, and so a candidate for measurement-based uncomputation. In this circuit it found exactly 12 such Toffoli groups; every other clean uncompute was already measurement-based. Credit: ours (harness probe, 2026-11-01). |
| Measurement-based uncomputation of a multi-input AND with a clean-wire ladder | HW-07, [Version 7](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/7) | HW-08 | A flag holding the AND of six bits, cleared by re-running a 16-Toffoli borrowed-wire chain, is instead measured in the X basis. Only when the outcome is 1 is a phase fix needed, (−1)^(y1⋯y6): it is applied by rebuilding the AND on two clean wires (3 CCX and one CCZ), each partial product then itself cleared by measurement-based uncomputation with a CZ or CCZ fix-up conditioned on its outcome. About 2.25 Toffolis expected per site instead of 16. Credit: measurement-based uncomputation of a temporary logical-AND is Gidney's (2018); the ladder and its application to these 12 sites are ours. |
| Porting a tweak to a new base by site census | HW-08, [Version 8](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/8) | — | A tweak found on one circuit often applies to a newer circuit from the same line. A census of the circuit file lists every site with the tweak's pattern (here, the three-input flag ANDs and six-input flags cleared by re-running a borrowed-wire chain) in about a minute, and the edits are generated from that list, so the port is mechanical and every site is accounted for. Re-running the original build code on the ported file and getting the same bytes checks the port. Credit: ours. |
| Conditionally clean flag computation with measurement-based cleanup | HW-09, Version 10 (this release) | — | A clean target and one or two clean scratch wires replace a 6- or 12-input dirty-borrow compute chain; input wires become scratch only in the branch where their values are known, are restored, and temporary products are measured with phase corrections. Credit: Khattar and Gidney for conditionally clean ancillae; our local relay for these exhaustively checked S12/S6 templates; manwithacat with Codex for this frontier port. |

## 4. All entries

One row per tweak, read from the dataset's `entries.csv`, which also holds the full verifier and draw results and every threshold, met or missed. ΔQ is the change in qubits and ΔT the change in mean Toffolis per shot. Expected GM is the change in the expected overall score, with the public frontier's circuits for the other two tasks, at the time of the entry.

In [ ]:
hw_csv = find("entries.csv")[0]
HW_DIR = hw_csv.parent
HW = list(csv.DictReader(open(hw_csv, newline="")))
show(pd.DataFrame([{
    "entry": e["id"], "date": e["date"], "task": e["task"], "change": e["title"],
    "base (author, circuit)": e["base_circuit"], "new circuit": e["candidate_sha12"],
    "ΔQ": int(e["delta_qubits"]), "ΔT per shot": f"{float(e['delta_toffoli_mean']):+,.1f}",
    "expected GM": f"{float(e['expected_gm_gain_pct']):+.3f}%",
    "failing draws (base → new)": (f"{e['draws_failing_base']} → {e['draws_failing_candidate']} of 48"
                                   if e["draws_failing_base"] else "no random draw"),
    "gates": ("missed: " + e["gates_missed"].split(":")[0]) if e["gates_missed"] else "all met",
    "base notebook": e["base_notebook"]} for e in HW]))
print(f"{len(HW)} entr{'y' if len(HW) == 1 else 'ies'} in {hw_csv.name} (sha256 {sha256(hw_csv)[:16]}...)")

In [ ]:
REQUIRED_ENTRIES = {'HW-09': '1698e864a3a638ef0569c438f01958731ebc164a917a4b84b8ff14598b83ab54'}
mounted = {e['id']: e['candidate_sha256'] for e in HW}
assert all(mounted.get(id_) == sha for id_, sha in REQUIRED_ENTRIES.items()), \
    "the attached Hedge dataset is stale: refresh to the release's pinned version"

## 5. What did not work

Ideas that failed are kept with the same care as those that worked, in the [VQCC Negative Results](https://www.kaggle.com/datasets/manwithacat/vqcc-chemistry-negative-results) dataset: one row per idea, with its evidence tier, why it failed and what would reopen it. Before trying something, check there.

- **Porting our failure fix to yoyobar's circuit gains nothing (HW-01).** Our Quantum Voodoo failure fix rebuilt decoder groups 4–7 as homogeneous chains with their poles dropped. yoyobar's circuit already has groups 3–7 built that way, so there is no Toffoli saving left and no chance-zero failure to remove (measured per group, local engine).
- **Releasing more idle wires cannot reach 887 (HW-01).** Releasing q866 or the 7-wire block cannot lower the peak, because 34 stretches at 888 use every live wire (24-bit multiplications; census, structural).
- **A column gauge alone does not beat Isabe11aM00re's (HW-03).** From Özer's weights, exact Jacobi sweeps and a smoothed-L1 L-BFGS from the identity, from Isabe11aM00re's frame and from 128 random kicks found nothing below Isabe11aM00re's frame (local prescreen). On Özer's C43 polish a gauge alone gained only 0.00073 in λ_eff (HW-04 run).
- **Our earlier claim that the fixed-word refit sits at a constrained optimum (NR-0080) was start-specific.** From Heliosli's weights it stalled; from Özer's, Isabe11aM00re's and Lera's it keeps improving (HW-03, HW-04).
- **Jumping error_1 straight to its limit fails (HW-04).** The refit's Gauss–Newton correction could not move the error_1 target from 13.3 to 18 or 20 mHa in one go; raising it in steps works.
- **yoyobar's 892-qubit DQI circuit does not reach 891 by cheap means (HW-06).** CNOT-only freeing covers 60 of 216 full-width stretches; releasing and recomputing the held epoch flags would cost about 1,900 Toffolis; clearing allocator artefacts leaves 132 stretches at 892. The normalize step needs a redesign (static model; see the HW-06 section).
- **Nor by releasing the carried pole flag (HW-07).** It is the one wire idle across every 892-qubit stretch, but it holds [N_k = 0] ∧ P with P the parity of the error weight, a global bit that no wire carries where it would be recomputed; rebuilding it costs about 1,872 Toffolis per shot against about 905 that a qubit is worth (bit-parallel classical simulation on 4,096 valid syndromes). 891 needs a new inversion kernel.

## 6. Credits

- **HW-01** (dqi): base circuit `73e1b187b288` by **yoyobar** ([notebook](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits) v8, Apache-2.0), on the decoder line of **SomeJoint (Agniv Banerjee)**; releasing a pole flag and recomputing it was first used publicly by **heliosli** on SomeJoint's decoder. Change: release-and-recompute (Quantum Voodoo line), the q852 release, the q878 reset and the handover erase, by manwithacat.
- **HW-03** (chemistry): weights by **Ahmed Berat Özer** (4eabe756bc36, [notebook](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression), Apache-2.0); per-rank column-gauge idea and the base point ec11a966fb64 by **Isabe11aM00re** ([notebook](https://www.kaggle.com/code/isabe11am00re/qcc-v47-submission) v3, Apache-2.0); word table by **Atakan Aldemir**; the (8, 143, 60) factorization by **SomeJoint (Agniv Banerjee)**; loaders by **Heliosli, Dmitrii Gluzdov and Atakan Aldemir**. Change: the refit + gauge alternation and the gauge search, by manwithacat.
- **HW-04** (chemistry): C42 factor/word table and loaders by **Lera (marculera)** (e9a9289ab06e, [notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v6, Apache-2.0). Change: the e1-spending refit + gauge alternation, by manwithacat. (The method was first run on Ahmed Berat Özer's C43 polish e9eb9336a5ec.)
- **HW-05** (chemistry): C40 factorization, 8-bit open-pattern words and all 15 loaders by **Lera (marculera)** ([notebook](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian) v9, Apache-2.0). Change: the fixed-word constrained W/H2_I refit by manwithacat (James Barlow), assisted by Codex (OpenAI). The refit method was developed in HW-03 and applied to C42 in HW-04.
- **HW-06** (dqi): base circuit `6d58bd6d0232` by **yoyobar** (with GPT-6 Astra; [notebook](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits) v11, Apache-2.0), the infinity-lead product reuse on **SomeJoint (Agniv Banerjee)**'s 892-qubit flag-carry circuit `bdee18489fac` ([notebook](https://www.kaggle.com/code/somejoint/quantum-circuits-23-77-local-gm-892-qubit-dqi) v25, Apache-2.0), whose flag carry follows **Ali Haydar Özdağ**'s flag carry. Change: measurement-based uncomputation at the 12 borrowed-wire flag ANDs, and the value tracker that found them, by manwithacat.
- **HW-07** (dqi): base circuit `c253111974fc`, our HW-06, on **yoyobar**'s `6d58bd6d0232` (with GPT-6 Astra; [notebook](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits) v11, Apache-2.0), the infinity-lead product reuse on **SomeJoint (Agniv Banerjee)**'s 892-qubit flag-carry circuit `bdee18489fac` ([notebook](https://www.kaggle.com/code/somejoint/quantum-circuits-23-77-local-gm-892-qubit-dqi) v25, Apache-2.0), whose flag carry follows **Ali Haydar Özdağ**'s flag carry. Reading Ali Haydar Özdağ's (ahoirg) public reproducible rebuild of SomeJoint's decoder sources ([notebook](https://www.kaggle.com/code/ahoirg/score-23-64254-reproducible-circuit-portfolio) v1) explained the normalize step's wires. Change: the two-wire measurement-based uncomputation of the 12 six-input flags, by manwithacat.
- **HW-08** (dqi): base circuit `b7ca9bce749d` by **SomeJoint (Agniv Banerjee)** ([notebook](https://www.kaggle.com/code/somejoint/quantum-circuits-23-85-local-gm-892-qubit-dqi) v26, Apache-2.0): GPT-6 Astra Pro's q26 decoder round, with Claude putting SomeJoint's flag carry, which follows **Ali Haydar Özdağ**'s, on it; the flag carry is as in `bdee18489fac` (**yoyobar**'s one-carry history). Change: HW-06's and HW-07's measurement-based uncomputation at 24 sites, found by our census tool, by manwithacat.
- **HW-09** (dqi): Base by **Akmal Xodarev (Busya PRIME)**: [33 QCC V8 members](https://www.kaggle.com/datasets/busyaprime/33-qcc-v8-members) v9, from [Exact DQI Rewrites](https://www.kaggle.com/code/busyaprime/23-910-local-gm-exact-dqi-rewrites-892-qubits) v15, Apache-2.0. The core decoder is **SomeJoint (Agniv Banerjee)**'s q26 892-qubit design; busyaprime adds exact product caching, credits **Ahmed Berat Özer** for that idea, and incorporates our HW-07 ladder. Our S12/S6 templates were developed in the earlier local relay, following **Khattar and Gidney**'s conditionally clean ancilla method; this port and verification are by **manwithacat (James Barlow), assisted by Codex (OpenAI)**.

The circuits are files of operations derived from other people's Apache-2.0 work; the dataset's NOTICE lists it. The tweaks are by manwithacat (James Barlow), made with substantial help from Claude (Anthropic) and Codex (OpenAI) under his direction; he chose the questions, approved each experiment and checked the claims.

## 7. How to reproduce

1. **Get the circuits.** Attach the [VQCC Hedge Wizardry dataset](https://www.kaggle.com/datasets/manwithacat/vqcc-hedge-wizardry). `entries.csv` gives each circuit's SHA-256 (`SHA256SUMS` lists every file; a chemistry entry's hash is its family hash). The base circuits are in the [VQCC public circuits dataset](https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits) under `circuits/<task>/<sha12>`.
2. **Verify on Kaggle.** Copy this notebook and run it (CPU, internet off). It installs our [kickmix fork](https://github.com/manwithacat/kickmix-macos) offline from the attached wheel and runs the unchanged official verifiers on the portfolio. The entry gate below checks every entry's cost against `entries.csv`. The verifier seeds every shot from the file's SHA-256, so the same bytes give the same cost at the default salt.
3. **Verify locally.** With kickmix built from our fork (macOS) or the official Linux build:
   - **HW-01**: run the official `dqi` verifier on `circuits/dqi/5faba4591853.kmb` with an empty salt: PASS, 888 qubits, cost 401,986,129,571.42.
   - **HW-03**: run the official `dfthcsos` verifier on `circuits/chemistry/e148cacdd4ad/` with an empty salt: PASS, 430 qubits, cost 337,651,009.06.
   - **HW-04**: run the official `dfthcsos` verifier on `circuits/chemistry/02f8f7d9d75a/` with an empty salt: PASS, 428 qubits, cost 331,025,488.43.
   - **HW-05**: run the official `dfthcsos` verifier on `circuits/chemistry/ff2b2733c1fc/` with an empty salt: PASS, 428 qubits, cost 326,959,774.66.
   - **HW-06**: run the official `dqi` verifier on `circuits/dqi/c253111974fc.kmb` with an empty salt: PASS, 892 qubits, cost 382,537,501,561.76.
   - **HW-07**: run the official `dqi` verifier on `circuits/dqi/1779b4e5967f.kmb` with an empty salt: PASS, 892 qubits, cost 384,137,374,413.52.
   - **HW-08**: run the official `dqi` verifier on `circuits/dqi/5e86f69a9611.kmb` with an empty salt: PASS, 892 qubits, cost 383,320,859,120.03.
   - **HW-09**: run the official `dqi` verifier on `circuits/dqi/1698e864a3a6.kmb` with an empty salt: PASS, 892 qubits, cost 378,829,899,786.41.
4. **DQI draws.** The 48 draws come from our local engine with fixed seeds shared by base and candidate. That engine is not published, so these numbers cannot be re-run from public files. The per-draw costs and errors are in `evidence/<entry>/draws_48.csv`.
5. **Read the change.** A `.kmb` circuit is a list of operations. The fork's circuit viewer (`tools/circuit-viewer/render.py`, including `ledger --compare`) shows where base and candidate differ. For chemistry entries, `evidence/<entry>/metrics.json` and `alternation_curve.csv` give the metrics of every stage.

## The submission this notebook makes

The hidden cells below:

1. pick the cheapest public circuit per task from the public dataset's manifest (chemistry: a verified assembly of public parts if that is cheaper than the best single family);
2. for each task, swap in the cheapest Hedge Wizardry entry if it is cheaper than the public pick, after checking its SHA-256;
3. install our kickmix fork offline;
4. run the unmodified official verifiers at full settings;
5. verify every entry's circuit and match its cost to `entries.csv` (the entry gate);
6. write `submission.zip` only if everything passes, with `check.json`, `entry_gates.json` and `receipt.json`.

In [ ]:
manifest = find("manifest.csv")[0]
DATA = manifest.parent
rows = list(csv.DictReader(open(manifest, newline="")))
print(f"manifest: {manifest} ({len(rows)} circuits, sha256 {sha256(manifest)[:16]}...)")

picks = {}
for task in REFERENCE:
    scored = rank_order([r for r in rows if r["task"] == task])  # tie group first, original first
    picks[task] = scored[0]
    best_cost = min(rank_cost(r) for r in scored)
    ties = [r["sha12"] for r in scored[1:] if rank_cost(r) <= best_cost * (1 + REL_TIE)]
    print(f"{task:9} {len(scored):2} candidates; picked {scored[0]['sha12']} "
          f"ranking cost {rank_cost(scored[0]):.6e} ({rank_basis(scored[0])}); "
          f"default-salt cost {float(scored[0]['local_task_score']):.6e}"
          + (f"; ties within noise, earlier public kept: {', '.join(ties)}" if ties else ""))

# Chemistry: a verified assembly of public parts replaces the best single family only if it is cheaper
ASSEMBLY = None
best_json = DATA / "parts" / "assembly_best.json"
if best_json.is_file():
    a = json.loads(best_json.read_text())
    single = float(picks["chemistry"]["local_task_score"])
    print(f"parts/assembly_best.json: verified {a['verified_cost']:.6e} ({a['verify_date']}) "
          f"vs best single family {picks['chemistry']['sha12']} {single:.6e}")
    if a.get("verified_status") == "PASS" and float(a["verified_cost"]) < single:
        ASSEMBLY = a

shutil.rmtree(SUB, ignore_errors=True)
SUB.mkdir(parents=True)
for task, r in picks.items():
    if task == "chemistry" and ASSEMBLY:
        fam = SUB / "dfthcsos"
        fam.mkdir()
        files = [(ASSEMBLY["npz"]["dataset_path"], ASSEMBLY["npz"]["sha256"], "submission.npz")]
        files += [(s["dataset_path"], s["file_sha256"], f"angle_k_{s['slot']}.kmb") for s in ASSEMBLY["slots"]]
        for path, digest, name in files:
            assert sha256(DATA / path) == digest, f"{path} does not match parts/assembly_best.json"
            shutil.copyfile(DATA / path, fam / name)
        authors = sorted({s["author"] for s in ASSEMBLY["slots"]})
        picks["chemistry"] = {
            "task": "chemistry", "sha12": "assembled", "sha256": family_sha256(fam),
            "qubits": ASSEMBLY["qubits"], "local_task_score": str(ASSEMBLY["verified_cost"]),
            "author": "assembled: loaders " + ", ".join(authors) + "; weights " + ASSEMBLY["npz"]["author"],
            "notebook_title": "assembly of public chemistry parts (parts/assembly_best.json)",
            "source_notebook_url": "https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits",
            "source_notebook_version": "", "release_date": ASSEMBLY["verify_date"],
            "core_author": ASSEMBLY["word_table_origin"]["author"],
            "change_summary": f"cheapest public circuit per angle loader with the best weights on word table "
                              f"{ASSEMBLY['table_key']}", "also_found_in": ""}
        print(f"chemistry: assembled from {len(files)} dataset files, every SHA-256 checked")
    elif task == "chemistry":
        src = DATA / "circuits" / "chemistry" / r["sha12"]
        assert family_sha256(src) == r["sha256"], f"chemistry family {r['sha12']} does not match the manifest"
        shutil.copytree(src, SUB / "dfthcsos")
    else:
        src = DATA / "circuits" / task / f"{r['sha12']}.kmb"
        assert sha256(src) == r["sha256"], f"{task} {r['sha12']} does not match the manifest"
        shutil.copyfile(src, SUB / f"{task}.kmb")
print("assembled and hash-checked:", sorted(p.name for p in SUB.iterdir()))

pd.set_option("display.max_colwidth", 80)
display(pd.DataFrame([{"task": t, "candidates": sum(1 for x in rows if x["task"] == t and x["local_task_score"]),
                       "picked": r["sha12"], "qubits": r.get("qubits"),
                       "ranking cost": rank_cost(r), "ranked by": rank_basis(r),
                       "local cost (default salt)": float(r["local_task_score"]), "author": r["author"],
                       "sha256 checked": True} for t, r in picks.items()]).set_index("task"))

In [ ]:
# Hedge Wizardry entries: verify each file's SHA-256, then let the cheapest entry per task replace the public
# pick only if it beats it on the same ranking cost (DQI: the 48-draw mean when the entry has one, against the
# public pick's rank_cost; others: the default-salt cost) by more than the tie threshold REL_TIE.
HW_FILES = {}
for e in HW:
    src = HW_DIR / e["dataset_path"]
    got = family_sha256(src) if e["task"] == "chemistry" else sha256(src)
    assert got == e["candidate_sha256"], f"{e['id']}: {src.name} does not match entries.csv"
    HW_FILES[e["id"]] = src
HW_USED = {}
for task in REFERENCE:
    def entry_cost(e):
        return float(e.get("draws_mean_cost_candidate") or e["local_task_score"]) if task == "dqi"                         else float(e["local_task_score"])
    mine = sorted((e for e in HW if e["task"] == task), key=entry_cost)
    if not mine:
        continue
    e, public = mine[0], picks[task]
    pub_cost = rank_cost(public)
    if not entry_cost(e) < pub_cost * (1 - REL_TIE):
        print(f"{task}: public {public['sha12']} ({pub_cost:.6e}, {rank_basis(public)}) stays; best entry "
              f"{e['id']} {e['candidate_sha12']} ({entry_cost(e):.6e}) does not beat it beyond the tie threshold")
        continue
    src = HW_FILES[e["id"]]
    if task == "chemistry":
        shutil.rmtree(SUB / "dfthcsos", ignore_errors=True)
        shutil.copytree(src, SUB / "dfthcsos")
        ASSEMBLY = None
    else:
        shutil.copyfile(src, SUB / f"{task}.kmb")
    picks[task] = {
        "task": task, "sha12": e["candidate_sha12"], "sha256": e["candidate_sha256"], "qubits": e["qubits"],
        "local_task_score": e["local_task_score"],
        "author": f"manwithacat (Hedge Wizardry {e['id']}) on {e['base_author']}'s {e['base_sha256'][:12]}",
        "notebook_title": f"VQCC Hedge Wizardry {e['id']}: {e['title']}",
        "source_notebook_url": "https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook",
        "source_notebook_version": "", "release_date": e["date"], "core_author": e["base_author"],
        "change_summary": e["card_summary"], "also_found_in": ""}
    HW_USED[task] = e["id"]
    print(f"{task}: Hedge Wizardry {e['id']} {e['candidate_sha12']} ({e['qubits']} qubits, local cost "
          f"{float(e['local_task_score']):.6e}) replaces public {public['sha12']} by {public['author']} "
          f"({pub_cost:.6e}, {rank_basis(public)}): {entry_cost(e) / pub_cost - 1:+.3%}")

In [ ]:
tool_dirs = [Path(p).parent for root in ("/kaggle/usr/lib", "/kaggle/input")
             for p in sorted(glob.glob(f"{root}/**/kickmix_tools.py", recursive=True))]
assert tool_dirs, "attach the kickmix_tools utility script"
sys.path.insert(0, str(tool_dirs[0]))
import kickmix_tools as kt
print("kickmix_tools", kt.__version__, "from", tool_dirs[0])

wheel_dir = find(WHEEL_GLOB)[0].parent
FORK_DIR = kt.install(str(wheel_dir), target="/tmp/kickmix_fork")
FORK_PROVENANCE = kt.provenance()
print("fork provenance:", FORK_PROVENANCE)
print("competition data:", kt.competition_dir())

In [ ]:
PATHS = {"invsqrt": SUB / "invsqrt.kmb", "dqi": SUB / "dqi.kmb", "chemistry": SUB / "dfthcsos"}
results, check_rows = {}, []
for task, circuit in PATHS.items():
    r = kt.verify(VERIFIER_TASK[task], circuit, build="fork")
    results[task] = r
    res = r["result"]
    listed = float(picks[task]["local_task_score"])
    check_rows.append({
        "task": task, "circuit": picks[task]["sha12"], "status": res["status"],
        "cost": res["score"], "source cost": listed, "cost / source": res["score"] / listed,
        "times cheaper than reference": REFERENCE[task] / res["score"],
        "qubits": res.get("num_qubits"), "Toffolis (mean)": res.get("toffoli_count"),
        "value errors": res.get("value_errors") if task == "dqi" else None,
        "phase errors": res.get("phase_errors") if task == "dqi" else None,
        "wall s": round(r["wall_seconds"], 1),
        "peak MB": round(r["peak_pss_sum_mb"] or r["peak_rss_sum_mb"] or 0)})
checks = pd.DataFrame(check_rows).set_index("task")
FORMATS = {"cost": "{:.6e}", "source cost": "{:.6e}", "cost / source": "{:.9f}",
           "times cheaper than reference": "{:.4f}", "qubits": "{:.0f}", "Toffolis (mean)": "{:,.1f}",
           "value errors": "{:.0f}", "phase errors": "{:.0f}", "wall s": "{:.1f}", "peak MB": "{:,.0f}"}
shown = checks.copy()
for col, fmt in FORMATS.items():  # value/phase errors exist for DQI only; blank cells show as —
    shown[col] = [fmt.format(v) if pd.notna(v) else "—" for v in checks[col]]
display(shown)
print("— = not applicable: only the DQI verifier counts value and phase errors.")
failed = [t for t, r in results.items() if r["result"]["status"] != "PASS"]
assert not failed, f"not writing a submission: {failed} did not pass"

ratios = {t: REFERENCE[t] / results[t]["result"]["score"] for t in REFERENCE}
gm = prod(ratios.values()) ** (1 / 3)
# DQI cost depends on the draw. The headline uses the DQI ranking cost (mean over a 48-draw sweep, or the
# zero-failure cost); gm is this run's single default-salt draw (2026-10-31: the grader uses its own hidden draw).
ratios_exp = dict(ratios, dqi=REFERENCE["dqi"] / rank_cost(picks["dqi"]))
gm_exp = prod(ratios_exp.values()) ** (1 / 3)
zf = picks['dqi'].get('dqi_zero_failure_cost')  # best case: no failed DQI shots in a draw
gm_zero = prod(dict(ratios, dqi=REFERENCE['dqi'] / float(zf)).values()) ** (1 / 3) if zf else None
# Error bounds (2026-11-01). Only DQI's cost depends on the draw: cost = R(k) x Q^2 x Toffolis, k the failed
# shots of the draw. The pick's 48-draw expected cost over its zero-failure cost is E[R(k)] / R(0); with k
# Poisson(lam) that fixes lam, so every file with a 48-draw sweep gets a measured final-draw risk.
R_K = [1.19126, 1.19673, 1.20139, 1.20558, 1.20946, 1.21311, 1.21657]  # R(k), 102,400 shots (verifier)
def _pois(lam):
    from math import exp, factorial
    p = [exp(-lam) * lam ** k / factorial(k) for k in range(len(R_K))]
    p[-1] += 1 - sum(p)
    return p
DQI_RISK = None
ec = picks['dqi'].get('dqi_expected_cost')
if zf and ec and float(ec) >= float(zf):
    target = float(ec) / float(zf)
    lo, hi = 0.0, 10.0
    for _ in range(80):  # E[R(k)]/R(0) rises with lam: bisect
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if sum(q * r / R_K[0] for q, r in zip(_pois(mid), R_K)) < target else (lo, mid)
    lam = (lo + hi) / 2
    pk = _pois(lam)
    gm_k = lambda k: prod(dict(ratios, dqi=REFERENCE['dqi'] / (float(zf) * R_K[k] / R_K[0])).values()) ** (1 / 3)
    DQI_RISK = {"failures_per_draw": lam, "p_none": pk[0], "p_one_or_more": 1 - pk[0],
                "p_two_or_more": 1 - pk[0] - pk[1], "gm_lucky": gm_k(0), "gm_one": gm_k(1), "gm_two": gm_k(2),
                "basis": "Poisson fit to the 48-draw expected cost / zero-failure cost (manifest)"}
print(f"expected geometric mean: {gm_exp:.5f} (DQI {rank_basis(picks['dqi'])}); "
      f"this run's single default-salt draw: {gm:.5f}")
checks.to_csv(WORK / "checks.csv")

### Entry gate

Every entry's circuit, verified here by the unchanged official verifier. An entry already in the portfolio reuses the run above, and any other is verified separately. Its cost must match the local record in `entries.csv` to 1e-9, which holds because the verifier seeds every shot from the file's SHA-256.

In [ ]:
ENTRY_GATES = []
for e in HW:
    task = e["task"]
    if picks[task]["sha256"] == e["candidate_sha256"]:
        res, how = results[task]["result"], "portfolio run"
    else:
        res, how = kt.verify(VERIFIER_TASK[task], HW_FILES[e["id"]], build="fork")["result"], "separate run"
    listed = float(e["local_task_score"])
    rel = abs(res["score"] / listed - 1)
    ENTRY_GATES.append({
        "entry": e["id"], "circuit": e["candidate_sha12"], "verifier run": how, "status": res["status"],
        "qubits": res.get("num_qubits"), "Toffolis (mean)": f"{res.get('toffoli_count', float('nan')):,.1f}",
        "cost (Kaggle)": f"{res['score']:.6e}", "entries.csv cost": f"{listed:.6e}",
        "matches to 1e-9": rel <= 1e-9, "in this submission": HW_USED.get(task) == e["id"]})
show(pd.DataFrame(ENTRY_GATES))
assert all(g["status"] == "PASS" for g in ENTRY_GATES), "an entry failed the official verifier"
assert all(g["matches to 1e-9"] for g in ENTRY_GATES), "a hosted entry cost differs from its local record"
(WORK / "entry_gates.json").write_text(json.dumps(ENTRY_GATES, indent=1, default=str))

In [ ]:
members = [("invsqrt.kmb", SUB / "invsqrt.kmb"), ("dqi.kmb", SUB / "dqi.kmb")]
members += [(f"dfthcsos/{p.relative_to(SUB / 'dfthcsos').as_posix()}", p)
            for p in sorted((SUB / "dfthcsos").rglob("*")) if p.is_file()]
with zipfile.ZipFile(WORK / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name, path in members:
        z.write(path, name)
(WORK / "check.json").write_text(json.dumps({
    "manifest_sha256": sha256(manifest), "submission_sha256": sha256(WORK / "submission.zip"),
    "geometric_mean_local": gm, "fork": FORK_PROVENANCE,
    "picks": {t: {k: picks[t][k] for k in ("sha256", "author", "notebook_title", "source_notebook_url",
                                            "release_date", "local_task_score")} for t in picks},
    "hedge_blend": globals().get("HEDGE_SELECTED", {}),
    "chemistry_assembly": ASSEMBLY,
    "results": {t: r["result"] for t, r in results.items()}}, indent=1, default=str))
shutil.rmtree(SUB)
print("wrote", WORK / "submission.zip", "and", WORK / "check.json")

### Verification receipt

The selected circuits, their sources, file hashes and verifier results, followed by checks that the costs reproduce their records and the run completes within two hours.

In [ ]:
WALL = time.monotonic() - T0
receipt = pd.DataFrame([{
    "task": TASK_NAMES[t], "circuit": picks[t]["sha12"], "sha256": picks[t]["sha256"][:16] + "…",
    "published by": picks[t]["author"], "version": picks[t]["source_notebook_version"],
    "verdict": results[t]["result"]["status"],
    "times cheaper than reference": REFERENCE[t] / results[t]["result"]["score"],
    "link": picks[t]["source_notebook_url"]} for t in REFERENCE])
show(receipt)
CHEM_PARTS = None
if ASSEMBLY:
    print("Chemistry was assembled from public parts: each angle loader, the weights and the word table's origin")
    CHEM_PARTS = pd.DataFrame(
        [{"part": f"angle loader {s['slot']}", "file": s["dataset_path"], "sha256": s["file_sha256"][:16] + "…",
          "published by": s["author"], "version": s.get("notebook_version", ""),
          "Toffolis per shot": s.get("toffoli_per_shot_source"), "link": s["notebook"]} for s in ASSEMBLY["slots"]]
        + [{"part": "weights (submission.npz)", "file": ASSEMBLY["npz"]["dataset_path"],
            "sha256": ASSEMBLY["npz"]["sha256"][:16] + "…", "published by": ASSEMBLY["npz"]["author"],
            "version": ASSEMBLY["npz"].get("notebook_version", ""), "Toffolis per shot": None,
            "link": ASSEMBLY["npz"]["notebook"]},
           {"part": f"word table {ASSEMBLY['table_key']} (origin)", "file": "parts/word_tables.csv",
            "sha256": "", "published by": ASSEMBLY["word_table_origin"]["author"], "version": "",
            "Toffolis per shot": None, "link": ASSEMBLY["word_table_origin"]["notebook"]}])
    show(CHEM_PARTS)
rel = {t: abs(results[t]["result"]["score"] / float(picks[t]["local_task_score"]) - 1) for t in REFERENCE}
G3_NAME = "G3 every cost within 1e-9 of its source record"
GATES = {
    "G1 every file matches its listed SHA-256": True,  # asserted when the files were assembled
    "G2 the official verifier passed all three": all(r["result"]["status"] == "PASS" for r in results.values()),
    G3_NAME: max(rel.values()) <= 1e-9,
    "G4 notebook wall time under 2 hours": WALL <= 7200,
}
show(pd.DataFrame([{"gate": g, "result": "pass" if ok else "FAIL"} for g, ok in GATES.items()]))
# G3: each cost reproduced exactly at the default salt (the local verifier seeds every shot from the file's
# SHA-256). This run's default-salt GM is one DQI draw; the grader draws with its own salt, so the headline's
# error bounds (expected, lucky, unlucky) are the fair summary, not this single draw.

# Headroom: how close each pick is to the verifier's pass/fail limits (from this run's verifier output)
E1_MAX = E2_MAX = 0.020          # Ha, both chemistry accuracy gates
GAP_FLOOR = 3.914898             # strict E_gap floor in the dfthcsos verifier
head = []
inv = results["invsqrt"]["result"]
w = inv.get("worst_error_over_tolerance")
if w is not None:
    head.append({"task": TASK_NAMES["invsqrt"], "check": "worst error / tolerance (must stay below 1)",
                 "value": f"{w:.5f}", "margin": f"{(1 - w) * 100:.3f}% of tolerance",
                 "tight": (1 - w) < 0.005})
dq = results["dqi"]["result"]
head.append({"task": TASK_NAMES["dqi"], "check": "failed shots on this file's draw (value / phase)",
             "value": f"{dq.get('value_errors')} / {dq.get('phase_errors')}",
             "margin": f"repetition factor R = {dq.get('repetition_factor', float('nan')):.5f}; "
                       "each extra failure costs about 0.46% of DQI cost",
             "tight": (dq.get("value_errors") or 0) + (dq.get("phase_errors") or 0) > 0})
ch = results["chemistry"]["result"].get("metrics", {})
for key, lim in (("error_1", E1_MAX), ("error_2", E2_MAX)):
    if key in ch:
        m = (lim - ch[key]) * 1000
        head.append({"task": TASK_NAMES["chemistry"], "check": f"{key} (must stay ≤ 20 mHa)",
                     "value": f"{ch[key] * 1000:.5f} mHa", "margin": f"{m:.5f} mHa", "tight": m < 0.01})
if "E_gap" in ch:
    m = ch["E_gap"] - GAP_FLOOR
    head.append({"task": TASK_NAMES["chemistry"], "check": "E_gap (must stay above 3.914898)",
                 "value": f"{ch['E_gap']:.7f}", "margin": f"{m:.2e}", "tight": m < 1e-4})
hd = pd.DataFrame(head)
hd["status"] = ["tight" if t else "comfortable" for t in hd.pop("tight")]
# Headroom: how close each pick is to failing the verifier; 'tight' means a tiny change in the file, the
# verifier or floating point could flip it.
show(hd)

# DQI final-salt risk: measured from the pick's 48-draw sweep (DQI_RISK, verify cell) and shown in the
# headline's error bounds; a DQI Salt Check output covering this file is kept in receipt.json when attached.
SALT = None
for f in sorted(glob.glob(f"{INPUT}/**/dqi_salt_check_summary_*.csv", recursive=True)):
    for r in csv.DictReader(open(f, newline="")):
        if r.get("circuit") == picks["dqi"]["sha12"]:
            SALT = dict(r, file=Path(f).name)
if not GATES[G3_NAME]:
    print("G3 failed: this run's verifier disagrees with a source record, so treat the local score with care.")
(WORK / "receipt.json").write_text(json.dumps({
    "gates": GATES, "geometric_mean_local": gm, "headroom": head, "dqi_salt_check": SALT, "dqi_risk": DQI_RISK, "wall_seconds": WALL, "relative_cost_difference": rel,
    "submission_sha256": sha256(WORK / "submission.zip"), "manifest_sha256": sha256(manifest),
    "rows": receipt.to_dict(orient="records"),
    "chemistry_parts": CHEM_PARTS.to_dict(orient="records") if CHEM_PARTS is not None else None},
    indent=1, default=str))